# Optimize performance in dataloading

In [5]:
import torch
from typing import Any, Dict, Optional, Tuple

from cv2 import transform
import torch
from lightning import LightningDataModule
from torch.utils.data import ConcatDataset, DataLoader, Dataset, random_split
from torchvision.transforms import transforms
import tonic
import numpy as np
import sys
sys.path.append("../src/utils")
sys.path.append("../src/data/")
sys.path.append("../src/data/components/")
sys.path.append("../src/models/components/")
import eventIO, event_represenations
from TOPSPIN import Hdf5Dataset
import fire_net
from topspin_datamodule import TopspinDataModule

import time
import pandas as pd

In [2]:
data_module = TopspinDataModule(
    data_dir="/data/lkolmar/datasets/topspin_fit_to_max/",
    time_window=5000,  # in us
    num_bins=10,
    sensor_size=(100, 100),
    train_val_test_split=(1294, 277, 277),  # n = 1848 (70%, 15%, 15%)
    batch_size=8,
    num_workers=1,
    pin_memory=True,
)
data_module.prepare_data()
data_module.setup()
dataloader = data_module.train_dataloader()

In [3]:
durations = []
n = 10
for i in range(n):
    start = time.time()
    batch = dataloader.__iter__().__next__()
    end = time.time()
    durations.append(end - start)
    print(f"Batch {i+1}/{n} took {end - start:.4f} seconds")
    #time.sleep(6)  # Sleep to simulate processing time

avg_duration = np.mean(durations)
print(f"Average duration for loading a batch: {avg_duration:.4f} seconds")
print(f"Total time for {n} batches: {np.sum(durations):.4f} seconds")

Batch 1/10 took 2.4170 seconds
Batch 2/10 took 2.7130 seconds
Batch 3/10 took 2.6969 seconds
Batch 4/10 took 2.4868 seconds
Batch 5/10 took 2.6943 seconds
Batch 6/10 took 2.4471 seconds
Batch 7/10 took 2.4442 seconds
Batch 8/10 took 2.6173 seconds
Batch 9/10 took 2.4626 seconds
Batch 10/10 took 2.6552 seconds
Average duration for loading a batch: 2.5634 seconds
Total time for 10 batches: 25.6345 seconds


## Average duration per batch: 4s
reducing num_workers helps and gives speedup up to 2s

## Benchmark the getitem function

In [4]:
events_struct = np.dtype(
    [("x", np.int16), ("y", np.int16), ("t", np.int64), ("p", np.int8)]
)

In [6]:
labels = pd.read_csv("/data/lkolmar/datasets/topspin_fit_to_max/config/labels.csv")

In [ ]:
def get_item(idx):
    index_str = str(idx).zfill(5)
    events = eventIO.load_hdf5(f"/data/lkolmar/datasets/topspin_fit_to_max/preprocessed/{index_str}/{index_str}_roi.hdf5")
    array = np.empty_like(events.get_x(), dtype=events_struct)
    array["x"] = events.get_x()
    array["y"] = events.get_y()
    array["t"] = events.get_ts()
    array["p"] = events.get_p()
    
    array = transforms(array)
    label = labels.loc(labels['index'] == idx, 'label').values[0]
    return array, label